In [21]:
import pandas as pd
import numpy as np
from datetime import date 
import glob
from glob import glob 
import os
import os.path as pa
import matplotlib.pyplot as plt
import sys

sys.path.append(os.path.abspath('00.src'))
import stats
import ettcdi
from get_data import query_climate_data

**Tahap 0:Mempersiapkan Folder Kerja**

In [22]:
workDir        = os.getcwd()
dataDir        = pa.join('../01.data')
outDir         = pa.join('../02.output')
srcDir         = pa.join('00.src')

obsDir         = pa.join(dataDir,"01.Obs_Homo")

os.makedirs(dataDir, exist_ok=True)
os.makedirs(outDir, exist_ok=True)
os.makedirs(srcDir, exist_ok=True)
os.makedirs(obsDir , exist_ok=True)
# rawDir       = pa.join(dataDir,'01.Raw')
# cleanDir     = pa.join(dataDir,"03.Clean")
# qcDir        = pa.join(dataDir,"04.Quality")

# reportDir    = pa.join(outDir,"01.Report")
# metaDir      = pa.join(outDir,"02.Metadata")
# indiceDir    = pa.join(outDir,"03.Indice")

**Tahap 1: Mengambil dataset dari database perubahan iklim**

In [23]:
# ==============================
# FUNGSI HITUNG ANOMALI BULANAN
# ==============================
start_date       = '1981-01-01'
end_date         = '2025-12-31'
# stations         = None
# provinces        = None
# regions          = None
# config_path      = None
# tdf     = query_climate_data(
#     parameters=['TEMPERATURE_AVG_C','TEMP_24H_TN_C','TEMP_24H_TX_C'],
#     sources='homo',
#     baseline='1981',
#     min_80pct_only=False,
#     min_80pct_baseline='1981',
#     start_date=start_date,
#     end_date=end_date,
#     stations=stations,
#     provinces=provinces,
#     regions=regions,
#     config_path=config_path)

# chdf   = query_climate_data(
#     parameters=['RAINFALL_24H_MM'],
#     sources='extended',
#     baseline='1981',
#     min_80pct_only=True,
#     min_80pct_baseline='1981',
#     start_date=start_date,
#     end_date=end_date,
#     stations=stations,
#     provinces=provinces,
#     regions=regions,
#     config_path=config_path)

# tdf   = tdf.drop(columns={'baseline','availability', 'meets_80pct', 'region', 'source'})
# chdf  = chdf.drop(columns={'baseline','availability', 'meets_80pct', 'region', 'source'})
# dfall = pd.concat([tdf, chdf], axis=0, ignore_index=True)

# Verifikasi tidak ada duplikasi kolom
# print("Duplikasi kolom:", dfall.columns[dfall.columns.duplicated()].tolist())
# print("Jumlah kolom:", len(dfall.columns))
# print("Kolom:", dfall.columns.tolist())

In [24]:
RAW_DIR   = '/mnt/dataset/02_REPO_GITHUB_FIRMAN/Developing_Climate_Observation_Dataset/data/05.Long_Format_Dataset'
PCT_DATA  = pd.read_csv(pa.join(RAW_DIR, '01.AVAILABILITY.csv'))
PCT_DATA  = PCT_DATA[(PCT_DATA['baseline'] == 1981) & (PCT_DATA['data_80%'] == True) & (PCT_DATA['parameter'] == 'RAINFALL_24H_MM')]
wmoids_ch = PCT_DATA['wmo_id'].unique().tolist()

In [25]:
RAW_DIR = '/mnt/dataset/02_REPO_GITHUB_FIRMAN/Developing_Climate_Observation_Dataset/data/05.Long_Format_Dataset'
tdf     = pd.read_csv(pa.join(RAW_DIR, '04.DATA_HOMO_DB.csv'), parse_dates=['time'])
tdf     = tdf[tdf['baseline'] == 1981].drop(columns='baseline')
chdf    = pd.read_csv(pa.join(RAW_DIR, '05.DATA_ROBI_DB.csv'), parse_dates=['time'])
chdf    = chdf[chdf['wmo_id'].isin(wmoids_ch)]
dfall   = pd.concat([tdf, chdf], axis=0, ignore_index=True)

**Tahap 2: Pivot parameter menjadi kolom**

In [26]:
# 1. HAPUS DUPLIKASI NAMA KOLOM (LANGKAH KRUSIAL!)
dfall = dfall.loc[:, ~dfall.columns.duplicated(keep='first')]
dfall = dfall[(dfall['time'] >= start_date) & (dfall['time'] <= end_date)]
# 2. VERIFIKASI TIDAK ADA DUPLIKASI LAGI
print("Kolom unik setelah cleanup:", dfall.columns.tolist())
print("Jumlah kolom:", len(dfall.columns))
# 3. PIVOT DENGAN METADATA YANG RELEVAN (sesuaikan dengan struktur data Anda)
metadata_cols = ['time', 'wmo_id', 'name', 'latitude', 'longitude', 'provinsi', 'kabupaten', 'elevasi']  # Tambahkan kolom yang benar-benar ada di data Anda
df_wide = dfall.pivot_table(
    index=metadata_cols,
    columns='parameter',
    values='value',
    aggfunc='first'
).reset_index()

# 4. HILANGKAN MULTI-INDEX
df_wide.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df_wide.columns]
df_wide = df_wide.rename(columns={'time_': 'time'})  # Jika ada kolom yang tergabung

# Sorting berdasarkan waktu dan stasiun
df_wide = df_wide.sort_values(by=['wmo_id', 'time']).reset_index(drop=True)

Kolom unik setelah cleanup: ['wmo_id', 'name', 'latitude', 'longitude', 'provinsi', 'kabupaten', 'elevasi', 'time', 'value', 'parameter', 'source']
Jumlah kolom: 11


**Tahap 3: Malakukan Spliting data menjadi Stasiun Berbeda**

In [27]:
groups        = df_wide.groupby('wmo_id')
for wmo_id, group_data in groups:
    file_name = f'{obsDir}/FKLIM_QC_DAILY_HOMO_EXTEND_{start_date}_{end_date}_{wmo_id}.csv'
    group_data.to_csv(file_name, index=False)